# ReAct-Agent: Agentic Workflow mit Tool-Zugang

## Was ist ein Agentic Workflow?

Ein **Agentic Workflow** bedeutet: Ein LLM trifft selbstständig Entscheidungen darüber, **welche Aktion es als nächstes ausführt** – anstatt nur eine Antwort zu generieren.

Das bekannteste Muster dafür heißt **ReAct** (Reason + Act):

```
Frage
  → LLM DENKT NACH (Reason): "Was brauche ich, um die Frage zu beantworten?"
  → LLM WÄHLT EIN TOOL (Act): z.B. get_revenue("Apple")
  → Tool läuft, Ergebnis zurück
  → LLM DENKT NACH (Reason): "Reicht das? Was fehlt noch?"
  → LLM WÄHLT NÄCHSTES TOOL ...
  → Finale Antwort
```

## Unser Beispiel

Ein **Finanzberater-Agent** beantwortet Fragen zu Tech-Unternehmen.  
Er hat Zugang zu 3 Tools:

| Tool | Was es tut |
|------|------------|
| `get_revenue(company)` | Gibt Umsatzdaten zurück |
| `calculate_growth(v1, v2)` | Berechnet Wachstumsrate in % |
| `get_market_share(company)` | Gibt Marktanteil zurück |

Das LLM entscheidet **selbst**, welche Tools es in welcher Reihenfolge aufruft.

## Schritt 1: Setup & Imports

Wir nutzen **OpenRouter** – eine API, die kostenlosen Zugang zu verschiedenen LLMs bietet.

In [ ]:
import os
import json
import requests
from getpass import getpass

# API-Key laden (sollte via ~/.zprofile gesetzt sein)
api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    api_key = getpass("OpenRouter API Key: ")
    os.environ["OPENROUTER_API_KEY"] = api_key

print("API-Key vorhanden:", bool(api_key))

## Schritt 2: Tools definieren

Tools sind einfache Python-Funktionen. Das LLM ruft sie **nicht direkt** auf – es sagt nur **welches Tool** es nutzen will und mit **welchen Argumenten**. Dann führt unser Code das Tool aus und gibt das Ergebnis zurück.

> **Wichtig:** Das LLM "sieht" nur den Tool-Namen und die Beschreibung – nicht den Code der Funktion selbst.

In [ ]:
# -------------------------------------------------------
# TOOL-DEFINITIONEN (Mock-Daten, kein echtes API-Call)
# -------------------------------------------------------

REVENUE_DATA = {
    "Apple":     {"2022": 394.3, "2023": 383.3, "2024": 391.0},
    "Microsoft": {"2022": 198.3, "2023": 211.9, "2024": 245.1},
    "NVIDIA":    {"2022":  26.9, "2023":  44.9, "2024":  60.9},
    "Meta":      {"2022": 116.6, "2023": 134.9, "2024": 164.5},
}

MARKET_SHARE_DATA = {
    "Apple":     14.2,
    "Microsoft": 18.7,
    "NVIDIA":     6.1,
    "Meta":       9.3,
}

def get_revenue(company: str, year: str = "2024") -> dict:
    """Gibt den Jahresumsatz eines Unternehmens in Mrd. USD zurück."""
    company = company.strip()
    if company not in REVENUE_DATA:
        return {"error": f"Unternehmen '{company}' nicht gefunden. Verfügbar: {list(REVENUE_DATA.keys())}"}
    if year not in REVENUE_DATA[company]:
        return {"error": f"Jahr '{year}' nicht verfügbar."}
    return {"company": company, "year": year, "revenue_usd_bn": REVENUE_DATA[company][year]}

def calculate_growth(value_old: float, value_new: float) -> dict:
    """Berechnet die prozentuale Wachstumsrate zwischen zwei Werten."""
    if value_old == 0:
        return {"error": "Startwert darf nicht 0 sein."}
    growth = round((value_new - value_old) / value_old * 100, 2)
    return {"value_old": value_old, "value_new": value_new, "growth_pct": growth}

def get_market_share(company: str) -> dict:
    """Gibt den Marktanteil eines Unternehmens in Prozent zurück."""
    company = company.strip()
    if company not in MARKET_SHARE_DATA:
        return {"error": f"Unternehmen '{company}' nicht gefunden. Verfügbar: {list(MARKET_SHARE_DATA.keys())}"}
    return {"company": company, "market_share_pct": MARKET_SHARE_DATA[company]}

# Tool-Registry: Name → Funktion (wird später vom Agent-Loop genutzt)
TOOLS = {
    "get_revenue": get_revenue,
    "calculate_growth": calculate_growth,
    "get_market_share": get_market_share,
}

print("Tools registriert:", list(TOOLS.keys()))

# Kurzer Test
print("\nTest get_revenue:", get_revenue("Apple", "2023"))
print("Test calculate_growth:", calculate_growth(383.3, 391.0))
print("Test get_market_share:", get_market_share("NVIDIA"))

## Schritt 3: Den Agenten mit Tools "ausstatten"

Das LLM muss wissen, welche Tools es nutzen kann. Dafür übergeben wir eine **Tool-Beschreibung** im System-Prompt.

Das LLM antwortet dann entweder mit:
- `{"action": "tool_name", "args": {...}}` → es will ein Tool aufrufen
- `{"action": "final_answer", "answer": "..."}` → es hat die Antwort

In [ ]:
SYSTEM_PROMPT = """Du bist ein Finanzberater-Agent. Du beantwortest Fragen zu Tech-Unternehmen.

Du hast Zugang zu diesen Tools:
- get_revenue(company, year): Gibt Jahresumsatz in Mrd. USD zurück. Jahre: 2022, 2023, 2024.
- calculate_growth(value_old, value_new): Berechnet Wachstumsrate in %.
- get_market_share(company): Gibt Marktanteil in % zurück.
  Verfügbare Unternehmen: Apple, Microsoft, NVIDIA, Meta

Antworte IMMER mit einem dieser zwei JSON-Formate (kein Text davor/danach):

Tool aufrufen:
{"action": "get_revenue", "args": {"company": "Apple", "year": "2024"}}

Finale Antwort:
{"action": "final_answer", "answer": "Hier steht deine Antwort in ganzen Sätzen."}
"""

def call_llm(messages: list) -> str:
    """Sendet Nachrichten an OpenRouter und gibt den Antworttext zurück."""
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
            "Content-Type": "application/json",
        },
        json={
            "model": "openrouter/auto",
            "messages": messages,
            "temperature": 0,
        },
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

def parse_llm_response(text: str) -> dict:
    """Extrahiert JSON aus der LLM-Antwort (robust gegen Markdown-Fences)."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1])
    return json.loads(text)

print("LLM-Hilfsfunktionen definiert.")

## Schritt 4: Der Agent-Loop (das Herzstück)

Hier passiert die Magie: Das LLM läuft in einer **Schleife**, bis es eine finale Antwort liefert.

```
while True:
    LLM denkt nach
    → will Tool?  → Tool ausführen → Ergebnis zurück in die Schleife
    → fertig?     → Antwort ausgeben, Schleife beenden
```

Jedes Tool-Ergebnis wird dem LLM als neue Nachricht mitgegeben – so baut sich der **Kontext** auf.

In [ ]:
def run_agent(question: str, max_steps: int = 6) -> str:
    """
    ReAct-Agent-Loop:
    1. Schickt die Frage ans LLM
    2. LLM wählt ein Tool → wir führen es aus → Ergebnis zurück ans LLM
    3. Wiederholen bis LLM "final_answer" liefert
    """
    # Nachrichtenverlauf (Kontext für das LLM)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    print(f"Frage: {question}")
    print("=" * 60)

    for step in range(1, max_steps + 1):
        # LLM entscheidet, was als nächstes zu tun ist
        raw_response = call_llm(messages)

        try:
            decision = parse_llm_response(raw_response)
        except json.JSONDecodeError:
            print(f"  [Fehler] LLM hat kein valides JSON geliefert: {raw_response}")
            break

        action = decision.get("action", "")

        # --- FALL 1: Finale Antwort ---
        if action == "final_answer":
            answer = decision.get("answer", "")
            print(f"\nFinale Antwort:\n{answer}")
            return answer

        # --- FALL 2: Tool-Aufruf ---
        if action in TOOLS:
            args = decision.get("args", {})
            print(f"  Schritt {step}: Tool '{action}' mit Args {args}")

            tool_result = TOOLS[action](**args)
            print(f"             → Ergebnis: {tool_result}")

            # Tool-Ergebnis in den Nachrichtenverlauf einfügen
            messages.append({"role": "assistant", "content": raw_response})
            messages.append({
                "role": "user",
                "content": f"Tool-Ergebnis von '{action}': {json.dumps(tool_result, ensure_ascii=False)}"
            })

        else:
            print(f"  [Unbekannte Action]: {action}")
            break

    return "Agent konnte keine Antwort finden."

print("Agent-Loop definiert und bereit.")

## Schritt 5: Agentenläufe starten

Jetzt stellen wir dem Agenten Fragen. Beobachtet, welche Tools er in welcher Reihenfolge aufruft!

**Frage 1:** Einfache Abfrage – nur 1 Tool nötig

In [ ]:
# Frage 1: Einfach – 1 Tool
run_agent("Wie hoch war der Umsatz von NVIDIA im Jahr 2024?")

**Frage 2:** Mehrstufig – Agent muss zuerst Umsätze holen, dann Wachstum berechnen (2-3 Tool-Aufrufe)

In [ ]:
# Frage 2: Mehrstufig – Agent braucht mehrere Tools
run_agent("Wie stark ist der Umsatz von Microsoft von 2022 auf 2024 gewachsen?")

**Frage 3:** Komplexe Vergleichsaufgabe – Agent muss mehrere Unternehmen abfragen und kombinieren

In [ ]:
# Frage 3: Komplex – mehrere Tools, mehrere Unternehmen
run_agent("Welches Unternehmen hat den höchsten Marktanteil – Apple oder Meta? Und wie hoch war deren Umsatz 2024?")

## Schritt 6: Multi-Agent – ein zweiter Agent prüft die Antwort

Jetzt fügen wir einen **Critic-Agenten** hinzu. Er bekommt die Antwort des ersten Agenten und bewertet sie kritisch.

Das entspricht dem Muster aus realen Systemen (z.B. LangGraph, AutoGen):
- **Agent 1 (Worker):** löst die Aufgabe
- **Agent 2 (Critic):** prüft Qualität und Vollständigkeit

In [ ]:
def critic_agent(question: str, answer: str) -> str:
    """
    Critic-Agent: Bewertet die Antwort des Worker-Agenten.
    Gibt strukturiertes Feedback zurück.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "Du bist ein kritischer Qualitätsprüfer für Finanzanalysen. "
                "Bewerte die gegebene Antwort auf eine Frage. "
                "Antworte in diesem JSON-Format:\n"
                '{"score": 1-10, "staerken": ["..."], "schwaechen": ["..."], "verbesserung": "..."}'
            ),
        },
        {
            "role": "user",
            "content": f"Frage: {question}\n\nAntwort: {answer}",
        },
    ]

    raw = call_llm(messages)
    try:
        feedback = json.loads(raw.strip().strip("```json").strip("```"))
    except json.JSONDecodeError:
        feedback = {"raw": raw}

    return feedback


def run_multi_agent(question: str) -> None:
    """Orchestrierung: Worker-Agent → Critic-Agent"""
    print("=" * 60)
    print("MULTI-AGENT WORKFLOW")
    print("=" * 60)

    # Agent 1: Worker löst die Aufgabe
    print("\n[Agent 1 – Worker] startet...\n")
    answer = run_agent(question)

    # Agent 2: Critic bewertet die Antwort
    print("\n[Agent 2 – Critic] bewertet die Antwort...\n")
    feedback = critic_agent(question, answer)

    print("\nCritic-Feedback:")
    print(json.dumps(feedback, ensure_ascii=False, indent=2))


# Starten!
run_multi_agent("Wie stark ist NVIDIA von 2022 auf 2024 gewachsen und wie hoch ist der Marktanteil?")

## Zusammenfassung

| Konzept | Was haben wir gesehen? |
|---|---|
| **Tool Use** | LLM entscheidet selbst, welches Tool es aufruft |
| **ReAct-Loop** | Denken → Handeln → Ergebnis → weiterdenken |
| **State / Kontext** | Alle bisherigen Tool-Ergebnisse bleiben im Nachrichtenverlauf |
| **Multi-Agent** | Worker-Agent löst, Critic-Agent bewertet |
| **Orchestrierung** | Python-Code steuert den Ablauf (nicht das LLM) |

### Mögliche Erweiterungen
- **Memory**: Zwischen verschiedenen Fragen Informationen speichern
- **Parallelisierung**: Mehrere Tools gleichzeitig aufrufen
- **Routing**: Orchestrator entscheidet, welcher Spezialist-Agent zuständig ist (z.B. Finanz-Agent vs. Rechts-Agent)
- **Human-in-the-Loop**: Mensch bestätigt bestimmte Aktionen bevor sie ausgeführt werden